# Assignment 4 — Customer, Product & Order Analysis

Datasets: `customers.csv`, `products.csv`, `orders.csv`

Objective: Combine separate datasets and perform complete business analysis using Pandas merging.

In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


## Part A — Understanding Individual Datasets

In [ ]:
# 1. Load all three CSV files
customers = pd.read_csv('../../data/customers.csv')
products = pd.read_csv('../../data/products.csv')
orders = pd.read_csv('../../data/orders.csv')

print("Datasets loaded successfully")

Datasets loaded successfully


In [3]:
# 2. Display the dimensions of each DataFrame
print(f"Customers: {customers.shape}")
print(f"Products: {products.shape}")
print(f"Orders: {orders.shape}")

Customers: (3000, 5)
Products: (100, 4)
Orders: (15000, 5)


In [4]:
# 3. Identify the columns in each dataset
print("Customers columns:", customers.columns.tolist())
print("Products columns:", products.columns.tolist())
print("Orders columns:", orders.columns.tolist())

Customers columns: ['Customer_ID', 'Customer_Name', 'City', 'Age', 'Gender']


Products columns: ['Product_ID', 'Product_Name', 'Category', 'Price']
Orders columns: ['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date']


In [5]:
# 4. Identify the common columns between datasets
cust_cols = set(customers.columns)
prod_cols = set(products.columns)
order_cols = set(orders.columns)

print(f"Customers ∩ Orders: {cust_cols & order_cols}")
print(f"Products ∩ Orders: {prod_cols & order_cols}")
print(f"Customers ∩ Products: {cust_cols & prod_cols}")

Customers ∩ Orders: {'Customer_ID'}
Products ∩ Orders: {'Product_ID'}
Customers ∩ Products: set()


In [6]:
# 5. Determine how the three datasets are related
print("Relationships:")
print("- Orders.Customer_ID → Customers.Customer_ID (many-to-one)")
print("- Orders.Product_ID → Products.Product_ID (many-to-one)")
print("- Customers and Products are linked through Orders")

Relationships:
- Orders.Customer_ID → Customers.Customer_ID (many-to-one)
- Orders.Product_ID → Products.Product_ID (many-to-one)
- Customers and Products are linked through Orders


## Part B — Data Cleaning

In [7]:
# 6. Check each DataFrame for missing values
for name, df in [('Customers', customers), ('Products', products), ('Orders', orders)]:
    missing = df.isnull().sum()
    print(f"\n{name} missing values:")
    print(missing[missing > 0])


Customers missing values:
Series([], dtype: int64)

Products missing values:
Series([], dtype: int64)

Orders missing values:
Series([], dtype: int64)


In [8]:
# 7. Check for duplicate records
for name, df in [('Customers', customers), ('Products', products), ('Orders', orders)]:
    dupes = df.duplicated().sum()
    print(f"{name} duplicates: {dupes}")

Customers duplicates: 0
Products duplicates: 0
Orders duplicates: 0


In [9]:
# 8. Check the validity of Customer_ID and Product_ID
print("Customer_ID in Customers:", customers['Customer_ID'].nunique(), "unique")
print("Customer_ID in Orders:", orders['Customer_ID'].nunique(), "unique")
print("Product_ID in Products:", products['Product_ID'].nunique(), "unique")
print("Product_ID in Orders:", orders['Product_ID'].nunique(), "unique")

Customer_ID in Customers:

 3000 unique
Customer_ID in Orders: 2976 unique
Product_ID in Products: 100 unique
Product_ID in Orders: 100 unique


In [10]:
# 9. Convert Order_Date into datetime
if 'Order_Date' in orders.columns:
    orders['Order_Date'] = pd.to_datetime(orders['Order_Date'], errors='coerce')
    print("Order_Date converted to datetime")
    print(f"Date range: {orders['Order_Date'].min()} to {orders['Order_Date'].max()}")

Order_Date converted to datetime
Date range: 2025-01-01 00:00:00 to 2028-06-03 22:00:00


In [11]:
# 10. Check whether every order corresponds to a valid customer
valid_customers = set(customers['Customer_ID'])
orders_customers = set(orders['Customer_ID'])
invalid_cust = orders_customers - valid_customers
print(f"Orders with invalid Customer_ID: {len(invalid_cust)}")
if len(invalid_cust) > 0:
    print(f"Invalid Customer_IDs: {invalid_cust}")

Orders with invalid Customer_ID: 0


In [12]:
# 11. Check whether every order corresponds to a valid product
valid_products = set(products['Product_ID'])
orders_products = set(orders['Product_ID'])
invalid_prod = orders_products - valid_products
print(f"Orders with invalid Product_ID: {len(invalid_prod)}")
if len(invalid_prod) > 0:
    print(f"Invalid Product_IDs: {invalid_prod}")

Orders with invalid Product_ID: 0


## Part C — Combining Data

In [13]:
# 12. Merge orders with customers
orders_customers = pd.merge(orders, customers, on='Customer_ID', how='left')
print(f"Orders + Customers merged: {orders_customers.shape}")
print("Columns:", orders_customers.columns.tolist())

Orders + Customers merged: (15000, 9)
Columns: ['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date', 'Customer_Name', 'City', 'Age', 'Gender']


In [14]:
# 13. Merge the resulting DataFrame with products
final_df = pd.merge(orders_customers, products, on='Product_ID', how='left')
print(f"Final merged DataFrame: {final_df.shape}")
print("Columns:", final_df.columns.tolist())

Final merged DataFrame: (15000, 12)
Columns: ['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date', 'Customer_Name', 'City', 'Age', 'Gender', 'Product_Name', 'Category', 'Price']


In [15]:
# 14. Create a final consolidated DataFrame
# The final DataFrame should contain information such as:
# Order_ID, Customer_ID, Customer_Name, City, Age, Gender,
# Product_ID, Product_Name, Category, Unit_Price, Quantity,
# Order_Date, Gross_Amount, Net_Amount

# Verify all required columns exist
required_cols = ['Order_ID', 'Customer_ID', 'Customer_Name', 'City', 'Age', 'Gender',
                 'Product_ID', 'Product_Name', 'Category', 'Unit_Price', 'Quantity',
                 'Order_Date']

for col in required_cols:
    if col in final_df.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col} MISSING")

✓ Order_ID
✓ Customer_ID
✓ Customer_Name
✓ City
✓ Age
✓ Gender
✓ Product_ID
✓ Product_Name
✓ Category
✗ Unit_Price MISSING
✓ Quantity
✓ Order_Date


In [16]:
# Bonus: Create derived columns for business analysis
# Note: products.csv has 'Price' column, not 'Unit_Price'
if 'Quantity' in final_df.columns and 'Price' in final_df.columns:
    final_df['Order_Value'] = final_df['Quantity'] * final_df['Price']
    print("Order_Value created")
    print(final_df['Order_Value'].describe())

Order_Value created
count     15000.000000
mean     148878.124753
std      120135.263877
min         651.660000
25%       51123.660000
50%      114353.600000
75%      224686.360000
max      493450.000000
Name: Order_Value, dtype: float64


In [17]:
# Quick business insights from consolidated data
print("=== BUSINESS INSIGHTS ===")

# Total revenue
if 'Net_Amount' in final_df.columns:
    print(f"Total Revenue: {final_df['Net_Amount'].sum():,.2f}")

# Top 5 customers by revenue
if 'Net_Amount' in final_df.columns and 'Customer_Name' in final_df.columns:
    top_customers = final_df.groupby('Customer_Name')['Net_Amount'].sum().sort_values(ascending=False).head(5)
    print("\nTop 5 Customers by Revenue:")
    print(top_customers)

# Top 5 products by revenue
if 'Net_Amount' in final_df.columns and 'Product_Name' in final_df.columns:
    top_products = final_df.groupby('Product_Name')['Net_Amount'].sum().sort_values(ascending=False).head(5)
    print("\nTop 5 Products by Revenue:")
    print(top_products)

# Revenue by city
if 'Net_Amount' in final_df.columns and 'City' in final_df.columns:
    city_revenue = final_df.groupby('City')['Net_Amount'].sum().sort_values(ascending=False)
    print("\nRevenue by City:")
    print(city_revenue)

=== BUSINESS INSIGHTS ===


## Part D — Business Calculations

In [18]:
# 15. Create: Order_Value = Price × Quantity
if 'Price' in final_df.columns and 'Quantity' in final_df.columns:
    final_df['Order_Value'] = final_df['Price'] * final_df['Quantity']
    print("Order_Value created")
    print(final_df['Order_Value'].describe())

Order_Value created
count     15000.000000
mean     148878.124753
std      120135.263877
min         651.660000
25%       51123.660000
50%      114353.600000
75%      224686.360000
max      493450.000000
Name: Order_Value, dtype: float64


In [19]:
# 16. Calculate total revenue
if 'Order_Value' in final_df.columns:
    total_revenue = final_df['Order_Value'].sum()
    print(f"Total Revenue: {total_revenue:,.2f}")

Total Revenue: 2,233,171,871.30


In [20]:
# 17. Calculate average order value
if 'Order_Value' in final_df.columns:
    avg_order = final_df['Order_Value'].mean()
    print(f"Average Order Value: {avg_order:.2f}")

Average Order Value: 148878.12

In [21]:
# 18. Find total revenue by product category
if 'Order_Value' in final_df.columns and 'Category' in final_df.columns:
    rev_by_cat = final_df.groupby('Category')['Order_Value'].sum().sort_values(ascending=False)
    print("Revenue by product category:")
    print(rev_by_cat)

Revenue by product category:


Category
Sports             5.426713e+08
Clothing           4.261489e+08
Home Appliances    3.313110e+08
Electronics        3.312926e+08
Books              3.050563e+08
Beauty             2.966918e+08
Name: Order_Value, dtype: float64


In [22]:
# 19. Find total revenue by city
if 'Order_Value' in final_df.columns and 'City' in final_df.columns:
    rev_by_city = final_df.groupby('City')['Order_Value'].sum().sort_values(ascending=False)
    print("Revenue by city:")
    print(rev_by_city)

Revenue by city:
City
Kolkata       3.155500e+08
Hyderabad     2.932639e+08
Coimbatore    2.831225e+08
Mumbai        2.803566e+08
Pune          2.797505e+08
Delhi         2.654541e+08
Bangalore     2.637059e+08
Chennai       2.519685e+08
Name: Order_Value, dtype: float64


In [23]:
# 20. Find the top 10 customers by spending
if 'Order_Value' in final_df.columns and 'Customer_Name' in final_df.columns:
    top10_cust = final_df.groupby('Customer_Name')['Order_Value'].sum().sort_values(ascending=False).head(10)
    print("Top 10 customers by spending:")
    print(top10_cust)

Top 10 customers by spending:
Customer_Name
Customer_19      3176078.87
Customer_2897    2528868.53
Customer_1915    2476932.74
Customer_2709    2400905.35
Customer_187     2394078.09
Customer_685     2352464.63
Customer_1297    2345815.39
Customer_259     2330070.43
Customer_921     2257528.60
Customer_271     2226186.64
Name: Order_Value, dtype: float64


In [24]:
# 21. Find the top 10 products by revenue
if 'Order_Value' in final_df.columns and 'Product_Name' in final_df.columns:
    top10_prod = final_df.groupby('Product_Name')['Order_Value'].sum().sort_values(ascending=False).head(10)
    print("Top 10 products by revenue:")
    print(top10_prod)

Top 10 products by revenue:
Product_Name
Product_74    47258003.68
Product_88    47213343.92
Product_3     47075130.00
Product_46    45000771.20
Product_52    44990386.56
Product_73    44897635.86
Product_68    44696626.32
Product_38    43658393.09
Product_54    40946977.89
Product_50    40756127.88
Name: Order_Value, dtype: float64


In [25]:
# 22. Find the most frequently purchased product
if 'Product_Name' in final_df.columns:
    most_freq = final_df['Product_Name'].value_counts().head(1)
    print(f"Most frequently purchased product: {most_freq.index[0]} ({most_freq.iloc[0]} times)")

Most frequently purchased product: Product_15 (183 times)


In [26]:
# 23. Find the city with the highest number of customers
if 'City' in final_df.columns:
    city_cust = final_df.groupby('City')['Customer_ID'].nunique().sort_values(ascending=False)
    print("Customers by city:")
    print(city_cust)
    top_city = city_cust.index[0]
    print(f"\nCity with most customers: {top_city}")

Customers by city:
City
Kolkata       411
Pune          385
Coimbatore    383
Hyderabad     370
Mumbai        370
Delhi         361
Bangalore     356
Chennai       340
Name: Customer_ID, dtype: int64

City with most customers: Kolkata


In [27]:
# 24. Find the city generating the highest revenue
if 'Order_Value' in final_df.columns and 'City' in final_df.columns:
    city_rev = final_df.groupby('City')['Order_Value'].sum().sort_values(ascending=False)
    top_city = city_rev.index[0]
    print(f"City generating highest revenue: {top_city} ({city_rev.iloc[0]:,.2f})")

City generating highest revenue: Kolkata (315,549,989.08)


## Part E — Customer Analysis

In [28]:
# 25. Divide customers into age groups
# 18–30, 31–45, 46–60, 61+
if 'Age' in final_df.columns:
    def age_group(age):
        if age <= 30:
            return '18-30'
        elif age <= 45:
            return '31-45'
        elif age <= 60:
            return '46-60'
        else:
            return '61+'
    final_df['Age_Group'] = final_df['Age'].apply(age_group)
    print("Age group distribution:")
    print(final_df['Age_Group'].value_counts())

Age group distribution:
Age_Group
46-60    4392
31-45    4231
18-30    3754
61+      2623
Name: count, dtype: int64


In [29]:
# 26. Calculate revenue generated by each age group
if 'Order_Value' in final_df.columns and 'Age_Group' in final_df.columns:
    rev_by_age = final_df.groupby('Age_Group')['Order_Value'].sum().sort_values(ascending=False)
    print("Revenue by age group:")
    print(rev_by_age)

Revenue by age group:
Age_Group
46-60    6.599237e+08
31-45    6.223524e+08
18-30    5.586278e+08
61+      3.922680e+08
Name: Order_Value, dtype: float64


In [30]:
# 27. Compare spending between male and female customers
if 'Order_Value' in final_df.columns and 'Gender' in final_df.columns:
    gender_spend = final_df.groupby('Gender')['Order_Value'].sum()
    print("Spending by gender:")
    print(gender_spend)

Spending by gender:
Gender
Female    1.078972e+09
Male      1.154200e+09
Name: Order_Value, dtype: float64


In [31]:
# 28. Identify high-value customers
if 'Order_Value' in final_df.columns and 'Customer_ID' in final_df.columns:
    cust_spend = final_df.groupby('Customer_ID')['Order_Value'].sum()
    # High-value: top 20%
    threshold = cust_spend.quantile(0.8)
    high_value = cust_spend[cust_spend >= threshold]
    print(f"High-value customers (top 20%): {len(high_value)}")
    print(high_value.sort_values(ascending=False).head(10))

High-value customers (top 20%): 596
Customer_ID
C00019    3176078.87
C02897    2528868.53
C01915    2476932.74
C02709    2400905.35
C00187    2394078.09
C00685    2352464.63
C01297    2345815.39
C00259    2330070.43
C00921    2257528.60
C00271    2226186.64
Name: Order_Value, dtype: float64


In [32]:
# 29. Determine the percentage of customers who have placed more than one order
if 'Customer_ID' in final_df.columns:
    order_counts = final_df.groupby('Customer_ID').size()
    multi_order = (order_counts > 1).sum()
    total_cust = len(order_counts)
    pct_multi = (multi_order / total_cust) * 100
    print(f"Customers with >1 order: {multi_order}/{total_cust} ({pct_multi:.1f}%)")

Customers with >1 order: 2881/2976 (96.8%)


## Part F — Advanced Analysis

In [33]:
# 30. Create a customer spending ranking
if 'Order_Value' in final_df.columns and 'Customer_ID' in final_df.columns:
    cust_spend = final_df.groupby('Customer_ID')['Order_Value'].sum().sort_values(ascending=False)
    final_df['Spending_Rank'] = final_df['Customer_ID'].map(cust_spend.rank(ascending=False, method='min').astype(int))
    print("Customer spending ranking created")
    print(cust_spend.head(10))

Customer spending ranking created
Customer_ID
C00019    3176078.87
C02897    2528868.53
C01915    2476932.74
C02709    2400905.35
C00187    2394078.09
C00685    2352464.63
C01297    2345815.39
C00259    2330070.43
C00921    2257528.60
C00271    2226186.64
Name: Order_Value, dtype: float64


In [34]:
# 31. Identify the top 10% of customers based on total spending
if 'Order_Value' in final_df.columns and 'Customer_ID' in final_df.columns:
    cust_spend = final_df.groupby('Customer_ID')['Order_Value'].sum()
    top10_pct = cust_spend.quantile(0.9)
    top10_cust = cust_spend[cust_spend >= top10_pct]
    print(f"Top 10% customers (spending >= {top10_pct:.2f}): {len(top10_cust)}")
    print(top10_cust.sort_values(ascending=False))

Top 10% customers (spending >= 1339423.81): 298


Customer_ID
C00019    3176078.87
C02897    2528868.53
C01915    2476932.74
C02709    2400905.35
C00187    2394078.09
             ...    
C00465    1344033.24
C00363    1343700.29
C02198    1343095.17
C01791    1340820.69
C00897    1340213.56
Name: Order_Value, Length: 298, dtype: float64


In [35]:
# 32. Find the most profitable product category for each city
if 'Order_Value' in final_df.columns and 'City' in final_df.columns and 'Category' in final_df.columns:
    city_cat_rev = final_df.groupby(['City', 'Category'])['Order_Value'].sum().unstack(fill_value=0)
    best_cat_per_city = city_cat_rev.idxmax(axis=1)
    print("Most profitable category per city:")
    print(best_cat_per_city)

Most profitable category per city:


City
Bangalore     Sports
Chennai       Sports
Coimbatore    Sports
Delhi         Sports
Hyderabad     Sports
Kolkata       Sports
Mumbai        Sports
Pune          Sports
dtype: str


In [36]:
# 33. Find the most popular product in each category
if 'Product_Name' in final_df.columns and 'Category' in final_df.columns:
    cat_prod = final_df.groupby(['Category', 'Product_Name']).size().unstack(fill_value=0)
    popular_per_cat = cat_prod.idxmax(axis=1)
    print("Most popular product per category:")
    print(popular_per_cat)

Most popular product per category:


Category
Beauty             Product_10
Books              Product_30
Clothing           Product_15
Electronics        Product_54
Home Appliances    Product_18
Sports             Product_48
dtype: str


In [37]:
# 34. Calculate monthly revenue
if 'Order_Value' in final_df.columns and 'Order_Date' in final_df.columns:
    final_df['Order_YearMonth'] = final_df['Order_Date'].dt.to_period('M')
    monthly_rev = final_df.groupby('Order_YearMonth')['Order_Value'].sum()
    print("Monthly revenue:")
    print(monthly_rev)

Monthly revenue:
Order_YearMonth
2025-01    53361220.93
2025-02    52963079.79
2025-03    56233288.12
2025-04    50923849.46
2025-05    57301343.61
2025-06    56460489.60
2025-07    54157148.69
2025-08    57144165.05
2025-09    53352193.95
2025-10    56630127.31
2025-11    59276825.16
2025-12    59630660.46
2026-01    56260151.26
2026-02    47689351.93
2026-03    55497703.15
2026-04    51306881.32
2026-05    51823753.71
2026-06    51618514.74
2026-07    54128777.26
2026-08    57438462.80
2026-09    55460907.01
2026-10    57171177.74
2026-11    56875351.32
2026-12    52435333.48
2027-01    56072466.01
2027-02    48032412.73
2027-03    51740476.06
2027-04    51673329.49
2027-05    55972559.14
2027-06    55791489.47
2027-07    57720544.86
2027-08    55334425.53
2027-09    53134726.53
2027-10    53646489.78
2027-11    53680865.96
2027-12    55292934.52
2028-01    52205366.58
2028-02    48474671.11
2028-03    55381632.73
2028-04    51636778.77
2028-05    57204319.10
2028-06     5035625.08
F

In [38]:
# 35. Identify the month with the highest revenue
if 'Order_Value' in final_df.columns and 'Order_Date' in final_df.columns:
    final_df['Order_YearMonth'] = final_df['Order_Date'].dt.to_period('M')
    monthly_rev = final_df.groupby('Order_YearMonth')['Order_Value'].sum()
    highest_month = monthly_rev.idxmax()
    highest_rev = monthly_rev.max()
    print(f"Highest revenue month: {highest_month} ({highest_rev:,.2f})")

Highest revenue month: 2025-12 (59,630,660.46)


## Summary

Assignment 4 complete. Three datasets merged into consolidated business view with derived metrics.